# 04 — Evaluation

Compare all fine-tuned models and LLM baselines on the **Russian annotated test set**.

**Sections**
1. Configuration
2. Run evaluation (inference on all models)
3. Results table
4. F1 comparison — weighted vs unweighted
5. Effect of K value
6. Effect of embedding space (raw vs PCA)
7. Metric heatmap
8. Confusion matrices
9. Bootstrap confidence intervals

In [1]:
import sys, os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation import evaluate_all, discover_models, compute_metrics, bootstrap_ci, get_confusion_matrix

sns.set_theme(style="whitegrid")

c:\Users\Alexandre\miniconda3\envs\faiss2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
CONFIG_PATH = "configs/datasets.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

PREPROCESSED_ROOT = cfg["preprocessing"]["output_root"]
TRAINING_ROOT     = cfg["training"]["output_root"]
EVAL_ROOT         = cfg["evaluation"]["output_root"]
MAX_LENGTH        = cfg["training"]["max_length"]

# Russian annotated test set — the only evaluation target
RUSSIAN_TEST_CSV  = os.path.join(PREPROCESSED_ROOT, "russian", "full.csv")
EXPERIMENT_NAME   = "main"
OUTPUT_DIR        = os.path.join(EVAL_ROOT, EXPERIMENT_NAME)

RUN_EVAL = True  # set True to run inference on all models

print("Russian test CSV:", RUSSIAN_TEST_CSV)
print("Training root   :", TRAINING_ROOT)
print("Output dir      :", OUTPUT_DIR)

Russian test CSV: outputs/1_preprocessed\russian\full.csv
Training root   : outputs/3_training
Output dir      : outputs/4_evaluation\main


## 2. Run Evaluation

Discovers all fine-tuned models from `outputs/3_training/`, runs inference on the Russian test set, and saves results.  
Set `RUN_EVAL = True` the first time. Subsequent runs load the cached `results.csv`.

In [3]:
results_path = os.path.join(OUTPUT_DIR, "results.csv")

# if RUN_EVAL or not os.path.exists(results_path):
if RUN_EVAL:
    results = evaluate_all(
        test_csv=RUSSIAN_TEST_CSV,
        training_root=TRAINING_ROOT,
        output_dir=OUTPUT_DIR,
        max_length=MAX_LENGTH,
        bootstrap=True,
        n_bootstrap=1000,
    )
else:
    print("Loading cached results from", results_path)
    results = pd.read_csv(results_path)

print(f"{len(results)} models evaluated")
print("Columns:", results.columns.tolist())

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 208.46it/s]


7 models evaluated
Columns: ['run_name', 'model_slug', 'train_dataset', 'density_tag', 'weighted', 'space', 'k', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'f1_ci_lower', 'f1_ci_upper']


## 3. Results Table

In [11]:
display_cols = ["run_name", "train_dataset", "density_tag", "weighted", "space", "k",
                "f1", "balanced_accuracy", "accuracy", "auc_roc"]
display_cols = [c for c in display_cols if c in results.columns]

display(
    results[display_cols]
    .sort_values("f1", ascending=False)
    .style.format({c: "{:.4f}" for c in ["f1", "balanced_accuracy", "accuracy", "auc_roc"] if c in display_cols})
    .background_gradient(subset=["f1"], cmap="YlGn")
)

,run_name,train_dataset,density_tag,weighted,space,k,f1,balanced_accuracy,accuracy,auc_roc
0,toxigen__density_k1000_ratio,toxigen,density_k1000_ratio,True,raw,1000.000000,0.5530,0.7692,0.7078,0.8504
1,toxigen__density_pca_k5_ratio,toxigen,density_pca_k5_ratio,True,pca,5.000000,0.5524,0.7641,0.7169,0.8354
2,toxigen__density_pca_k1000_ratio,toxigen,density_pca_k1000_ratio,True,pca,1000.000000,0.5466,0.7614,0.7068,0.8361
3,toxigen__density_pca_k100_ratio,toxigen,density_pca_k100_ratio,True,pca,100.000000,0.5443,0.7567,0.7108,0.8289
4,toxigen__no_density,toxigen,no_density,False,nan,nan,0.5441,0.7661,0.6888,0.8554
5,toxigen__density_k100_ratio,toxigen,density_k100_ratio,True,raw,100.000000,0.5436,0.7650,0.6898,0.8564
6,toxigen__density_k5_ratio,toxigen,density_k5_ratio,True,raw,5.000000,0.5412,0.7631,0.6867,0.8539


## 4. F1 Comparison — Density-Weighted vs Unweighted

In [5]:
df = results.dropna(subset=["f1", "weighted"]).copy()
df["weight_label"] = df["weighted"].map({True: "Density-weighted", False: "Unweighted"})

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="weight_label", y="f1", ax=ax, palette=["#e74c3c", "#95a5a6"])
sns.stripplot(data=df, x="weight_label", y="f1", ax=ax, color="black", alpha=0.5, size=4)
ax.set_xlabel("")
ax.set_ylabel("F1 Score")
ax.set_title("F1 — density-weighted vs unweighted training")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "weighted_vs_unweighted.png"), dpi=150)
plt.show()

C:\Users\Alexandre\AppData\Local\Temp\ipykernel_1904\2691979275.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="weight_label", y="f1", ax=ax, palette=["#e74c3c", "#95a5a6"])
C:\Users\Alexandre\AppData\Local\Temp\ipykernel_1904\2691979275.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Effect of K Value

In [6]:
df_k = results.dropna(subset=["k", "f1"]).copy()
df_k["k"] = df_k["k"].astype(int)

if not df_k.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=df_k, x="k", y="f1", ax=ax)
    sns.stripplot(data=df_k, x="k", y="f1", ax=ax, color="black", alpha=0.5, size=4)
    ax.set_xlabel("K value")
    ax.set_ylabel("F1 Score")
    ax.set_title("F1 per K value (density-weighted models only)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plots", "f1_by_k.png"), dpi=150)
    plt.show()
else:
    print("No density-weighted models found yet.")

C:\Users\Alexandre\AppData\Local\Temp\ipykernel_1904\2172159401.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Effect of Embedding Space (Raw vs PCA)

In [7]:
df_space = results.dropna(subset=["space", "f1"]).copy()

if not df_space.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    # Box per space
    sns.boxplot(data=df_space, x="space", y="f1", ax=axes[0], palette=["#3498db", "#e67e22"])
    sns.stripplot(data=df_space, x="space", y="f1", ax=axes[0], color="black", alpha=0.5, size=4)
    axes[0].set_title("F1 by embedding space")
    axes[0].set_xlabel("")

    # Line per K, grouped by space
    df_grouped = df_space.groupby(["k", "space"])["f1"].mean().reset_index()
    sns.lineplot(data=df_grouped, x="k", y="f1", hue="space", marker="o", ax=axes[1])
    axes[1].set_title("Mean F1 per K × space")
    axes[1].set_xlabel("K value")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plots", "f1_space_vs_k.png"), dpi=150)
    plt.show()
else:
    print("No density-weighted models found yet.")

C:\Users\Alexandre\AppData\Local\Temp\ipykernel_1904\1868115419.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_space, x="space", y="f1", ax=axes[0], palette=["#3498db", "#e67e22"])
C:\Users\Alexandre\AppData\Local\Temp\ipykernel_1904\1868115419.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Metric Heatmap

In [8]:
metric_cols = [c for c in ["accuracy", "balanced_accuracy", "f1", "auc_roc"] if c in results.columns]
heatmap_df = (
    results.dropna(subset=["f1"])
    .set_index("run_name")[metric_cols]
    .astype(float)
    .sort_values("f1", ascending=False)
)

fig, ax = plt.subplots(figsize=(8, max(4, len(heatmap_df) * 0.45)))
sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlGnBu",
            vmin=0, vmax=1, linewidths=0.5, annot_kws={"size": 8}, ax=ax)
ax.set_title("All metrics — Russian annotated test set")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "metrics_heatmap.png"), dpi=150)
plt.show()

C:\Users\Alexandre\AppData\Local\Temp\ipykernel_1904\209332687.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Confusion Matrices

Saved per-model PNGs are in `outputs/4_evaluation/main/confusion/`.  
Display the top-3 models here.

In [9]:
from PIL import Image
import glob

confusion_dir = os.path.join(OUTPUT_DIR, "confusion")
top3 = results.dropna(subset=["f1"]).head(3)["run_name"].tolist()

fig, axes = plt.subplots(1, len(top3), figsize=(5 * len(top3), 4))
if len(top3) == 1:
    axes = [axes]

for ax, run_name in zip(axes, top3):
    safe = run_name.replace("/", "_").replace("\\", "_")
    img_path = os.path.join(confusion_dir, f"{safe}.png")
    if os.path.exists(img_path):
        ax.imshow(Image.open(img_path))
        ax.axis("off")
        ax.set_title(run_name[:40], fontsize=8)
    else:
        ax.set_title(f"{run_name[:30]} — not found")
        ax.axis("off")

plt.tight_layout()
plt.show()

C:\Users\Alexandre\AppData\Local\Temp\ipykernel_1904\3879816421.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Bootstrap Confidence Intervals

F1 CIs for all models, sorted by point estimate.

In [10]:
ci_cols = ["run_name", "f1", "f1_ci_lower", "f1_ci_upper"]
ci_cols = [c for c in ci_cols if c in results.columns]

if "f1_ci_lower" in results.columns:
    df_ci = results[ci_cols].dropna().sort_values("f1", ascending=True)

    fig, ax = plt.subplots(figsize=(8, max(4, len(df_ci) * 0.4)))
    ax.barh(df_ci["run_name"], df_ci["f1"], color="#3498db", alpha=0.7)
    ax.errorbar(
        df_ci["f1"], df_ci["run_name"],
        xerr=[df_ci["f1"] - df_ci["f1_ci_lower"], df_ci["f1_ci_upper"] - df_ci["f1"]],
        fmt="none", color="black", capsize=3, linewidth=1,
    )
    ax.set_xlabel("F1 (95% CI)")
    ax.set_title("F1 with Bootstrap Confidence Intervals")
    ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plots", "f1_bootstrap_ci.png"), dpi=150)
    plt.show()
else:
    print("Bootstrap CIs not computed — re-run with bootstrap=True.")

C:\Users\Alexandre\AppData\Local\Temp\ipykernel_1904\141569523.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
